# 陨石图像生成 -- Assignment 3
## WGAN-GP 训练 Notebook

**目标**：使用 WGAN-GP 在 2682 张陨石图像上训练生成模型，生成 1000 张新图像用于 FID 评估。

**方案 C（折中）**：128x128, epochs=300, n_critic=3, BATCH_SIZE x2

**目录结构**（评估前确认）：
```
assignment3/
├── evaluate_fid.py
├── inception_v3_google-0cc3c7bd.pth
├── meteorite/           # 2682 张训练图像
├── generated_pictures/  # [本 notebook 生成] 1000 张
└── evaluation_results/  # [evaluate_fid.py 生成]
```


In [ ]:
from __future__ import annotations

import os
import random
import time
from datetime import datetime
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # 无 GUI 后端，适配远端服务器
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.utils import make_grid
from tqdm import tqdm

# ---------- GPU / 环境信息 ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"  显存: {mem_gb:.1f} GB")
    print(f"  CUDA: {torch.version.cuda}")
else:
    print("  [WARNING] 未检测到 GPU，训练将非常慢!")


In [ ]:
SCRIPT_DIR = Path.cwd()
DATA_DIR = SCRIPT_DIR / "meteorite"
GENERATED_DIR = SCRIPT_DIR / "generated_pictures"
CHECKPOINT_DIR = SCRIPT_DIR / "checkpoints"
GRID_PATH = SCRIPT_DIR / "generated_grid.png"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

assert DATA_DIR.is_dir(), f"未找到 {DATA_DIR}, 请确认 notebook 在 assignment3 根目录"
n_imgs = sum(1 for p in DATA_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)
print(f"数据目录: {DATA_DIR}")
print(f"图像数量: {n_imgs}")


In [ ]:
class MeteoriteDataset(Dataset):
    """加载 meteorite/ 图像, 缩放到统一尺寸, 归一化到 [-1, 1]"""

    def __init__(self, image_dir: Path, image_size: int) -> None:
        self.image_paths = sorted(
            p for p in Path(image_dir).iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        )
        if not self.image_paths:
            raise FileNotFoundError(f"目录中未找到图像: {image_dir}")

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10, fill=1.0),     # ±10° 旋转, 白色填充
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> torch.Tensor:
        img = Image.open(self.image_paths[idx]).convert("RGB")
        return self.transform(img)


In [ ]:
class Generator(nn.Module):
    """DCGAN 生成器: latent_dim -> image_size x image_size RGB
    支持 64x64 和 128x128 分辨率"""

    def __init__(self, latent_dim: int = 100, feature_maps: int = 64,
                 channels: int = 3, image_size: int = 64) -> None:
        super().__init__()
        self.latent_dim = latent_dim
        fm = feature_maps

        layers = [
            # (latent_dim, 1, 1) -> (fm*16, 4, 4)
            nn.ConvTranspose2d(latent_dim, fm * 16, 4, 1, 0, bias=False),
            nn.BatchNorm2d(fm * 16),
            nn.ReLU(True),

            # -> (fm*8, 8, 8)
            nn.ConvTranspose2d(fm * 16, fm * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(fm * 8),
            nn.ReLU(True),

            # -> (fm*4, 16, 16)
            nn.ConvTranspose2d(fm * 8, fm * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(fm * 4),
            nn.ReLU(True),

            # -> (fm*2, 32, 32)
            nn.ConvTranspose2d(fm * 4, fm * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(fm * 2),
            nn.ReLU(True),
        ]

        if image_size >= 128:
            layers += [
                # -> (fm, 64, 64)
                nn.ConvTranspose2d(fm * 2, fm, 4, 2, 1, bias=False),
                nn.BatchNorm2d(fm),
                nn.ReLU(True),

                # -> (channels, 128, 128)
                nn.ConvTranspose2d(fm, channels, 4, 2, 1, bias=False),
                nn.Tanh(),
            ]
        else:
            layers += [
                # -> (channels, 64, 64)
                nn.ConvTranspose2d(fm * 2, channels, 4, 2, 1, bias=False),
                nn.Tanh(),
            ]

        self.main = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.main(z.view(z.size(0), self.latent_dim, 1, 1))


In [ ]:
class Critic(nn.Module):
    """WGAN-GP Critic with SpectralNorm: image_size x image_size RGB -> 单值分数
    使用 SpectralNorm 替代 InstanceNorm，梯度更平滑，训练更稳定"""

    def __init__(self, feature_maps: int = 64, channels: int = 3,
                 image_size: int = 64) -> None:
        super().__init__()
        fm = feature_maps

        layers = []
        if image_size >= 128:
            layers += [
                nn.utils.spectral_norm(nn.Conv2d(channels, fm, 4, 2, 1, bias=False)),
                nn.LeakyReLU(0.2, inplace=True),
            ]
            mult_seq = [1, 2, 4, 8, 16, 16]
        else:
            layers += [
                nn.utils.spectral_norm(nn.Conv2d(channels, fm, 4, 2, 1, bias=False)),
                nn.LeakyReLU(0.2, inplace=True),
            ]
            mult_seq = [1, 2, 4, 8, 16]

        in_ch = fm
        for mult in mult_seq[1:-1]:
            out_ch = fm * mult
            layers += [
                nn.utils.spectral_norm(nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)),
                nn.LeakyReLU(0.2, inplace=True),
            ]
            in_ch = out_ch

        layers += [
            nn.Conv2d(in_ch, 1, 4, 1, 0, bias=True),
        ]

        self.main = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.main(x).view(-1)


In [ ]:
def init_weights(model: nn.Module) -> None:
    """权重初始化: Normal(0, 0.02) for Conv, Normal(1, 0.02) for Norm"""
    for m in model.modules():
        if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)):
            try:
                nn.init.normal_(m.weight, mean=1.0, std=0.02)
                nn.init.constant_(m.bias, 0.0)
            except AttributeError:
                pass


In [ ]:
def generate_fixed_noise(latent_dim: int, n: int, device: torch.device) -> torch.Tensor:
    """生成固定噪声用于训练中可视化进度"""
    return torch.randn(n, latent_dim, device=device)


def gradient_penalty(critic, real, fake, device):
    """WGAN-GP 梯度惩罚"""
    batch_size = real.size(0)
    epsilon = torch.rand(batch_size, 1, 1, 1, device=device)
    interpolated = epsilon * real + (1 - epsilon) * fake
    interpolated.requires_grad_(True)

    critic_interpolated = critic(interpolated)

    grad = torch.autograd.grad(
        outputs=critic_interpolated,
        inputs=interpolated,
        grad_outputs=torch.ones_like(critic_interpolated),
        create_graph=True,
        retain_graph=True,
    )[0]

    grad_norm = grad.view(batch_size, -1).norm(2, dim=1)
    return ((grad_norm - 1) ** 2).mean()


def save_grid(generator, fixed_noise, path, device):
    """保存 3x3 九宫格预览图"""
    generator.eval()
    with torch.no_grad():
        fake = generator(fixed_noise).detach().cpu()
    grid = make_grid(fake, nrow=3, normalize=True, value_range=(-1, 1))
    vutils.save_image(grid, path)
    generator.train()


def generate_evaluation_images(generator, latent_dim, output_dir, device,
                               batch_size=50, total=1000):
    """生成 1000 张评估图像到 generated_pictures/"""
    output_dir.mkdir(parents=True, exist_ok=True)
    for old in output_dir.iterdir():
        if old.suffix.lower() in IMAGE_EXTENSIONS:
            old.unlink()

    generator.eval()
    generated = 0
    pbar = tqdm(total=total, desc="生成评估图像")
    with torch.no_grad():
        while generated < total:
            n = min(batch_size, total - generated)
            z = torch.randn(n, latent_dim, device=device)
            fake = generator(z).detach().cpu()
            for i in range(fake.size(0)):
                vutils.save_image(
                    fake[i],
                    output_dir / f"{generated + i + 1:04d}.jpg",
                    normalize=True,
                    value_range=(-1, 1),
                )
            generated += n
            pbar.update(n)
    pbar.close()
    generator.train()
    print(f"已生成 {total} 张图像 -> {output_dir}")

    if device.type == "cuda":
        torch.cuda.empty_cache()


In [ ]:
# ==================== 超参数配置 (调参 v4: EMA + 加容量 + 数据增强) ====================
# 改动: FEATURE_MAPS 96→128 / 加 EMA / 加 Rotation+ColorJitter / 修复 total_memory
IMAGE_SIZE = 128       # 生成/训练图像分辨率
LATENT_DIM = 256       # 噪声向量维度
FEATURE_MAPS = 128     # 基础特征图数 (96→128, 进一步提升容量)
EPOCHS = 600           # 训练轮数
LR_G = 0.0002          # 生成器学习率
LR_D = 0.00005         # Critic 学习率
BETA1 = 0.0            # Adam beta1
BETA2 = 0.9            # Adam beta2
LAMBDA_GP = 5          # 梯度惩罚系数
N_CRITIC = 2           # 每训练 1 次 G 训练 N 次 D
EMA_DECAY = 0.999      # EMA 衰减率 (用于生成评估图像)
NUM_WORKERS = 4        # DataLoader 进程数
SEED = 42              # 随机种子
SAVE_INTERVAL = 25     # checkpoint 保存间隔 (epoch)

# 根据 GPU 显存自动调整 BATCH_SIZE (x2 加速)
if device.type == "cuda":
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if IMAGE_SIZE >= 128:
        BATCH_SIZE = 128 if mem_gb >= 20 else 64 if mem_gb >= 10 else 32
    else:
        BATCH_SIZE = 256 if mem_gb >= 20 else 128 if mem_gb >= 10 else 64
    if mem_gb >= 16:
        BATCH_SIZE = min(BATCH_SIZE * 2, 256)
        print(f"  [方案C] BATCH_SIZE 翻倍 -> {BATCH_SIZE}")
else:
    BATCH_SIZE = 16

print(f"图像分辨率: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Batch Size:  {BATCH_SIZE} (auto)")
print(f"Epochs:      {EPOCHS}")
print(f"LR_G: {LR_G}  LR_D: {LR_D}")
print(f"LATENT_DIM: {LATENT_DIM}  FEATURE_MAPS: {FEATURE_MAPS}")
print(f"lambda_gp: {LAMBDA_GP}  n_critic: {N_CRITIC}  ema_decay: {EMA_DECAY}")
print(f"Save Interval: {SAVE_INTERVAL} epochs")


In [ ]:
# ==================== 可复现性 ====================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ==================== 数据加载 ====================
dataset = MeteoriteDataset(DATA_DIR, IMAGE_SIZE)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    drop_last=True,
)
print(f"训练集: {len(dataset)} 张图像, {len(dataloader)} 个 batch/epoch")

sample_batch = next(iter(dataloader))
print(f"Batch shape: {sample_batch.shape}  (应为 [{BATCH_SIZE}, 3, {IMAGE_SIZE}, {IMAGE_SIZE}])")


In [ ]:
# ==================== 模型初始化 ====================
netG = Generator(LATENT_DIM, FEATURE_MAPS, 3, IMAGE_SIZE).to(device)
netD = Critic(FEATURE_MAPS, 3, IMAGE_SIZE).to(device)
netG.apply(init_weights)
netD.apply(init_weights)

print(f"Generator 参数量: {sum(p.numel() for p in netG.parameters()):,}")
print(f"Critic 参数量:    {sum(p.numel() for p in netD.parameters()):,}")

# ==================== 优化器 (WGAN-GP: 两个独立优化器) ====================
optimizerG = optim.Adam(netG.parameters(), lr=LR_G, betas=(BETA1, BETA2))
optimizerD = optim.Adam(netD.parameters(), lr=LR_D, betas=(BETA1, BETA2))
# 不再需要 nn.BCELoss()

# ==================== EMA (指数滑动平均) ====================
ema_netG = Generator(LATENT_DIM, FEATURE_MAPS, 3, IMAGE_SIZE).to(device)
ema_netG.load_state_dict(netG.state_dict())
ema_netG.eval()  # EMA 只用于推理
for p in ema_netG.parameters():
    p.requires_grad_(False)

fixed_noise = generate_fixed_noise(LATENT_DIM, 9, device)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("模型已初始化, 准备训练 (WGAN-GP + EMA)...")


In [ ]:
# ==================== WGAN-GP 训练循环 ====================
# WGAN loss 解读: D_loss 应该趋近于 0 (Wasserstein 距离估计)
# G_loss 会波动, 这是正常的 — 不像 BCE 那样博弈
g_losses = []
d_losses = []
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    epoch_g_loss = 0.0
    epoch_d_loss = 0.0
    n_batches = 0

    for real in pbar:
        batch_size_curr = real.size(0)
        real = real.to(device)
        n_batches += 1

        # ---- 训练 Critic N_CRITIC 次 ----
        d_loss_sum = 0.0
        for _ in range(N_CRITIC):
            z = torch.randn(batch_size_curr, LATENT_DIM, device=device)
            fake = netG(z).detach()

            netD.zero_grad()
            critic_real = netD(real)
            critic_fake = netD(fake)
            gp = gradient_penalty(netD, real.data, fake.data, device)

            loss_D = -critic_real.mean() + critic_fake.mean() + LAMBDA_GP * gp
            loss_D.backward()
            optimizerD.step()
            d_loss_sum += loss_D.item()

        epoch_d_loss += d_loss_sum / N_CRITIC

        # ---- 训练 Generator 1 次 ----
        netG.zero_grad()
        z = torch.randn(batch_size_curr, LATENT_DIM, device=device)
        fake = netG(z)
        loss_G = -netD(fake).mean()

        loss_G.backward()
        optimizerG.step()
        epoch_g_loss += loss_G.item()

        pbar.set_postfix(D=f"{loss_D.item():.3f}", G=f"{loss_G.item():.3f}")

    # ---- EMA 更新 ----
    with torch.no_grad():
        for ema_p, g_p in zip(ema_netG.parameters(), netG.parameters()):
            ema_p.data.mul_(EMA_DECAY).add_(g_p.data, alpha=1 - EMA_DECAY)

    avg_d = epoch_d_loss / n_batches
    avg_g = epoch_g_loss / n_batches

    # NaN 检测
    if np.isnan(avg_d) or np.isnan(avg_g):
        print(f"[ERROR] NaN detected at epoch {epoch}! 训练中断，请尝试降低 LR。")
        break

    g_losses.append(avg_g)
    d_losses.append(avg_d)
    epoch_time = time.time() - epoch_start
    elapsed = time.time() - start_time

    if epoch % 10 == 0 or epoch == 1 or epoch == EPOCHS:
        eta = (elapsed / epoch) * (EPOCHS - epoch)
        print(f"[Epoch {epoch:3d}/{EPOCHS}]  "
              f"D_loss: {avg_d:.4f}  G_loss: {avg_g:.4f}  "
              f"耗时: {epoch_time:.0f}s  ETA: {eta/60:.0f}min")

    if epoch % SAVE_INTERVAL == 0 or epoch == EPOCHS:
        torch.save({
            "epoch": epoch,
            "generator_state_dict": netG.state_dict(),
            "ema_generator_state_dict": ema_netG.state_dict(),
            "discriminator_state_dict": netD.state_dict(),
            "optimizerG_state_dict": optimizerG.state_dict(),
            "optimizerD_state_dict": optimizerD.state_dict(),
            "g_losses": g_losses,
            "d_losses": d_losses,
        }, CHECKPOINT_DIR / f"checkpoint_epoch_{epoch:04d}.pth")

        save_grid(netG, fixed_noise, GRID_PATH, device)
        print(f"  -> 已保存 checkpoint & 九宫格 (epoch {epoch})")

    if device.type == "cuda":
        torch.cuda.empty_cache()

total_time = time.time() - start_time
print(f"\\n训练完成! 总耗时: {total_time/60:.1f} min ({total_time/3600:.2f} h)")


In [ ]:
# ==================== 损失曲线 ====================
plt.figure(figsize=(10, 4))
plt.plot(g_losses, label="Generator Loss", alpha=0.8)
plt.plot(d_losses, label="Critic Loss (Wasserstein)", alpha=0.8)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("WGAN-GP Training Loss (Critic loss -> 0 = better)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
loss_path = SCRIPT_DIR / "training_loss.png"
plt.savefig(loss_path, dpi=100)
plt.close()
print(f"损失曲线已保存: {loss_path}")


In [ ]:
# ==================== 生成 1000 张评估图像 (使用 EMA 权重) ====================
best_checkpoint = sorted(CHECKPOINT_DIR.glob("checkpoint_epoch_*.pth"))[-1]
print(f"加载 checkpoint: {best_checkpoint.name}")
checkpoint = torch.load(best_checkpoint, map_location=device, weights_only=True)
# 优先加载 EMA 权重，其次普通权重
if "ema_generator_state_dict" in checkpoint:
    ema_netG.load_state_dict(checkpoint["ema_generator_state_dict"])
    print("  -> 使用 EMA 权重生成")
else:
    netG.load_state_dict(checkpoint["generator_state_dict"])
    print("  -> 使用普通权重生成")

generate_evaluation_images(ema_netG, LATENT_DIM, GENERATED_DIR, device)

gen_count = sum(1 for p in GENERATED_DIR.iterdir()
                if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
print(f"generated_pictures/ 图像数量: {gen_count}")
assert gen_count == 1000, f"期望 1000 张, 实际 {gen_count} 张!"
print("\n现在可以运行: python evaluate_fid.py")


In [ ]:
# ==================== 报告用 3x3 九宫格 ====================
# 从生成图像中随机选 9 张 -- 建议手动替换为视觉效果最好的 9 张

all_generated = sorted(
    p for p in GENERATED_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
)

selected = random.sample(all_generated, 9)
images = [Image.open(p) for p in selected]

grid_img = Image.new("RGB", (IMAGE_SIZE * 3, IMAGE_SIZE * 3), "white")
for idx, img in enumerate(images):
    row, col = divmod(idx, 3)
    grid_img.paste(img.resize((IMAGE_SIZE, IMAGE_SIZE)),
                   (col * IMAGE_SIZE, row * IMAGE_SIZE))

report_grid_path = SCRIPT_DIR / "report_3x3_grid.png"
grid_img.save(report_grid_path)
print(f"报告九宫格已保存: {report_grid_path}")
print("  (提示: 可在本地手动挑选最佳 9 张替换此图)")


## 下一步

1. **评估 FID**：运行下一个 Cell（自动运行 `evaluate_fid.py`），或手动在终端运行 `python evaluate_fid.py`
2. **挑选最佳 9 张**：下载 `generated_pictures/` 到本地，手动挑选视觉效果最好的 9 张替换九宫格
3. **下载关键文件**：
   - `evaluation_results/fid_metrics.json` — FID 分数
   - `report_3x3_grid.png` — 九宫格
   - `training_loss.png` — 损失曲线
   - `generated_grid.png` — 训练过程预览
4. **撰写报告** (`report.pdf`)
5. **打包代码**：运行最后一个 Cell 自动打包 `code.zip`

In [ ]:
# ==================== 自动评估 FID ====================
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "evaluate_fid.py"],
    capture_output=True, text=True, cwd=str(SCRIPT_DIR)
)
print(result.stdout)
if result.returncode != 0:
    print("[STDERR]", result.stderr)
    print("[WARNING] FID 评估未成功，请手动运行: python evaluate_fid.py")
else:
    # 读取 FID 分数
    import json
    metrics_path = SCRIPT_DIR / "evaluation_results" / "fid_metrics.json"
    if metrics_path.is_file():
        metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
        print(f"\\nFID = {metrics['fid']:.4f}")


In [ ]:
# ==================== 打包提交代码 ====================
import zipfile

zip_path = SCRIPT_DIR / "student_id_name_assignment3_code.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in SCRIPT_DIR.iterdir():
        if f.is_file() and f.suffix in {".py", ".ipynb"}:
            zf.write(f, f.name)
            print(f"  + {f.name}")

print(f"\\n代码已打包: {zip_path}")
print("提交前请重命名为: student_id_name_assignment3_code.zip")
